# Missing Values: Diagnose Before You Fill

Measure missingness and compare defensible strategies for handling it.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Inspect missing values by column and row.
- Distinguish deletion from imputation.
- Use numeric and categorical fill strategies carefully.
- Record the assumptions behind a cleaning decision.

Missingness is not just a technical nuisance. It may indicate a failed sensor, a skipped question, an unavailable service, or a meaningful event.

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "district": ["Central", "North", "North", "West", "Central", "West"],
    "income": [1200, np.nan, 980, 1500, np.nan, 1100],
    "visits": [4, 5, np.nan, 2, 7, np.nan],
    "service_type": ["Library", "Clinic", None, "Library", "Clinic", "Library"],
})
display(df)
print("Missing count:\n", df.isna().sum())
print("Missing percent:\n", (df.isna().mean() * 100).round(1))

Missing count:
 district        0
income          2
visits          2
service_type    1
dtype: int64
Missing percent:
 district         0.0
income          33.3
visits          33.3
service_type    16.7
dtype: float64


,district,income,visits,service_type
0,Central,1200.0,4.0,Library
1,North,NaN,5.0,Clinic
2,North,980.0,NaN,None
3,West,1500.0,2.0,Library
4,Central,NaN,7.0,Clinic
5,West,1100.0,NaN,Library


## Deletion and imputation options
- `dropna()` can be reasonable when few rows are incomplete and their absence is not systematic.
- Mean imputation is sensitive to outliers; the median is more robust.
- A missing category can be explicitly labeled `"Unknown"` when that distinction matters.
- Never replace missing values with zero unless zero has the real-world meaning you intend.

In [2]:
complete_income = df.dropna(subset=["income"])
print("Rows retained:", len(complete_income), "of", len(df))

clean = df.copy()
clean["income_median_filled"] = clean["income"].fillna(clean["income"].median())
clean["visits_missing"] = clean["visits"].isna().astype("int8")
clean["service_type"] = clean["service_type"].fillna("Unknown")
display(clean)

Rows retained: 4 of 6


,district,income,visits,service_type,income_median_filled,visits_missing
0,Central,1200.0,4.0,Library,1200.0,0
1,North,NaN,5.0,Clinic,1150.0,0
2,North,980.0,NaN,Unknown,980.0,1
3,West,1500.0,2.0,Library,1500.0,0
4,Central,NaN,7.0,Clinic,1150.0,0
5,West,1100.0,NaN,Library,1100.0,1


## Group-aware imputation and validation
Group-specific fill values can preserve differences across groups, but they should only be used when the grouping has a defensible relationship to the missing variable. Track which rows were imputed.

In [3]:
clean = df.copy()
group_medians = clean.groupby("district")["income"].transform("median")
clean["income_filled"] = clean["income"].fillna(group_medians)
clean["income_was_missing"] = clean["income"].isna()
display(clean[["district", "income", "income_filled", "income_was_missing"]])
print("Still missing:", clean["income_filled"].isna().sum())

Still missing: 0


,district,income,income_filled,income_was_missing
0,Central,1200.0,1200.0,False
1,North,NaN,980.0,True
2,North,980.0,980.0,False
3,West,1500.0,1500.0,False
4,Central,NaN,1200.0,True
5,West,1100.0,1100.0,False


## Practice
Count missing values, then fill numeric values with medians and missing category labels with `"Unreported"`. Preserve the original DataFrame.

In [4]:
source = df.copy()
cleaned = source.copy()
print(source.isna().sum())
for col in ["income", "visits"]:
    cleaned[col] = cleaned[col].fillna(cleaned[col].median())
cleaned["service_type"] = cleaned["service_type"].fillna("Unreported")
display(cleaned)
print("Original still has missing values:", source.isna().any().any())

district        0
income          2
visits          2
service_type    1
dtype: int64
Original still has missing values: True


,district,income,visits,service_type
0,Central,1200.0,4.0,Library
1,North,1150.0,5.0,Clinic
2,North,980.0,4.5,Unreported
3,West,1500.0,2.0,Library
4,Central,1150.0,7.0,Clinic
5,West,1100.0,4.5,Library
